# DS207 Final Project - Predicting Readmission Rate for Diabetic Patients Using Machine Learning
#### Contributor: Colin Frishberg (cpfrish@berkeley.edu)

### Notebook Structure:

1.  Create compelling **data visualizations**.
2.  **Input:** Load raw and processed data from data files.
3.  Build, train, and evaluate the **Transformer Model**.
4.  **Output:** Save the visualizations to html files in the '/results' directory
5.  **Output:** Save the trained Transformer model to `/results/transformer.h5`.
6.  **Output:** Save the statistics from the transformer model to `/results/stats_transformer.csv`.

## 1. Notebook Imports

### 1.1 Import basic and necessary libraries:

In [ ]:
# Basic imports
import numpy as np
import warnings
import pandas as pd
from matplotlib import pyplot as plt
from tensorflow import keras
import altair as alt

alt.data_transformers.enable("vegafusion")

# Compress all warnings
warnings.filterwarnings("ignore")

### 1.2 Import datasets:

In [ ]:
# Load raw data set
df = pd.read_csv("../data/diabetic_data.csv")

# Load cleaned data set (from colin_dev)
df_cleaned_dev = pd.read_csv("../data/diabetic_data_cleaned_encoded.csv")

# Load train, val, test splits (from Rahil's notebook)
df_train = pd.read_csv("../results/train.csv")
df_val = pd.read_csv("../results/val.csv")
df_test = pd.read_csv("../results/test.csv")

# Load colin_dev data splits (for model development)
df_train_dev = pd.read_csv("../data/X_train_scaled.csv")
df_val_dev = pd.read_csv("../data/X_val_scaled.csv")
df_test_dev = pd.read_csv("../data/X_test_scaled.csv")
df_y_train_dev = pd.read_csv("../data/y_train.csv")
df_y_val_dev = pd.read_csv("../data/y_val.csv")
df_y_test_dev = pd.read_csv("../data/y_test.csv")

# Combine train, val, test into a single dataframe for visualization
df_combined_clean = pd.concat([df_train, df_val, df_test], axis=0)

In [ ]:
# Display shape of raw and cleaned data sets
print(f"Raw data set shape: {df.shape}")
print(f"Cleaned data set shape: {df_cleaned_dev.shape}")
print(f"Combined data set shape: {df_combined_clean.shape}")
print(f"Train split shape: {df_train.shape}")
print(f"Validation split shape: {df_val.shape}")
print(f"Test split shape: {df_test.shape}")

## 2. Visualization

Visualization requirements: Include multiple detailed plots that effectively communicate your data insights: ensure all plots have properly labeled x and y axes; include descriptive titles; add legends where appropriate; consider using multiple plot types (histograms, scatter plots, box plots, heatmaps, etc.) to highlight different aspects of your data; accompany each visualization with interpretations of what the patterns reveal.

In [ ]:
# Install vegafusion for Altair visualizations
# %pip install "vegafusion[embed]>=1.5.0"
# Install vl-convert-python for Altair visualizations
# %pip install "vl-convert-python>=1.6.0"

#### Kernel Density Estimates

In [ ]:
continuous_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "age_mid",
]

In [ ]:
def create_kde_plots(df, kde_vars, ncols=2):
    """
    Generates and combines KDE plots into a grid for given features

    Args:
        df (pd.Dataframe)
        kde_vars(list): List of columns to plot
        ncols(int): Number of columns in the grid

    Returns:
        alt.vconcat: Grid of KDE plots
    """
    charts = []
    for var in kde_vars:
        chart = (
            alt.Chart(df)
            .transform_density(
                density=var,
                as_=[var, "Density"],  # The output fields for the value and its density
                groupby=["readmitted"],
            )
            .mark_area(opacity=0.5)
            .encode(
                x=alt.X(f"{var}:Q", title=var.replace("_", " ").title()),
                y=alt.Y("Density:Q"),
                # Color by the 'readmitted' class aka target variable
                color=alt.Color("readmitted:N", title="Readmitted Class"),
            )
            .properties(
                title=f"Distribution of {var.replace('_', ' ').title()}",
                width=300,
                height=200,
            )
        )
        charts.append(chart)

        # Grid creations
        rows = [
            alt.hconcat(*charts[i : i + ncols]) for i in range(0, len(charts), ncols)
        ]
    # Combine all created charts vertically and return them
    return alt.vconcat(*rows)

#### Box Plots

In [ ]:
# Box Plots for distributions of key predictors to outcome
def create_box_plots(df, continuous_features):
    """
    Generates and combines box plots into a grid for given features

    Args:
        df (pd.Dataframe)
        continuous_features(list): List of columns to plot
    """
    box_plots = []
    for feature in continuous_features:
        # Create box plot for each feature against readmission
        chart = (
            alt.Chart(df)
            .mark_boxplot()
            .encode(
                x=alt.X(
                    "readmitted:N", title="Readmitted Class (0: NO, 1: >30, 2: <30)"
                ),
                y=alt.Y(f"{feature}:Q", title=feature.replace("_", " ").title()),
                color=alt.Color("readmitted:N", title="Readmitted Class").scale(
                    scheme="category20"
                ),
                tooltip=["readmitted", feature],
            )
            .properties(
                title=f"Box Plot of {feature.replace('_', ' ').title()} by Readmission Status",
                width=300,
                height=200,
            )
        )
        box_plots.append(chart)

    # Arrange the box plots in a grid
    ncols = 3
    rows = [
        alt.hconcat(*box_plots[i : i + ncols]) for i in range(0, len(box_plots), ncols)
    ]
    alt.vconcat(*rows)
    return alt.vconcat(*rows)

#### Violin Plots

In [ ]:
def create_violin_plots(df, continuous_features):
    """
    Generates and combines violin plots into a grid for given features

    Args:
        df (pd.Dataframe)
        continuous_features(list): List of columns to plot
    """
    # Create violin plots for each continuous feature

    violin_plots = []
    for feature in continuous_features:
        chart = (
            alt.Chart(df)
            .transform_density(
                density=feature, as_=[feature, "density"], groupby=["readmitted"]
            )
            .mark_area(orient="horizontal")
            .encode(
                y=alt.Y(f"{feature}:Q", title=feature.replace("_", " ").title()),
                x=alt.X(
                    "density:Q",
                    stack="center",
                    impute=None,
                    title=None,
                    axis=alt.Axis(labels=False, values=[0], grid=False, ticks=True),
                ),
                color=alt.Color(
                    "readmitted:N", legend=alt.Legend(title="Readmitted")
                ).scale(scheme="tableau20"),
            )
            .properties(
                title=f"Distribution of {feature.replace('_', ' ').title()} by Readmission",
                width=300,
                height=250,
            )
        )
        violin_plots.append(chart)

    # Arrange the violin plots in a grid
    ncols = 4
    rows_violin = [
        alt.hconcat(*violin_plots[i : i + ncols])
        for i in range(0, len(violin_plots), ncols)
    ]
    final_violin_chart = alt.vconcat(*rows_violin)
    return final_violin_chart


## 3. Model Developments

In [ ]:
def build_transformer_model(
    input_shape: int,
    num_classes: int = 2,
    embedding_dim: int = 192,
    num_heads: int = 8,
    key_dim: int = 24,
    ffn_dim: int = 2048,
    dense_dim: int = 64,
):
    """
    Build a transformer model for readmission prediction.

    Args:
        input_shape: Number of input features
        num_classes: Number of output classes
        embedding_dim: Dimension of embedding layer
        num_heads: Number of attention heads
        key_dim: Dimension of key in attention
        ffn_dim: Dimension of feed-forward network
        dense_dim: Dimension of final dense layer

    Returns:
        Compiled Keras model
    """

    inputs = keras.Input(shape=(input_shape,))

    # Embedding layer
    x = keras.layers.Dense(
        embedding_dim, activation="relu", kernel_regularizer=keras.regularizers.l2(0.01)
    )(inputs)
    x = keras.layers.Reshape((1, embedding_dim))(x)

    # Dropout layer
    x = keras.layers.Dropout(0.1)(x)

    # Transformer block
    attention_output = keras.layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim
    )(x, x)
    x = keras.layers.Add()([x, attention_output])
    x = keras.layers.LayerNormalization()(x)

    # Feed-forward network
    ffn_output = keras.layers.Dense(
        ffn_dim, activation="relu", kernel_regularizer=keras.regularizers.l2(0.01)
    )(x)
    ffn_output = keras.layers.Dense(
        embedding_dim, kernel_regularizer=keras.regularizers.l2(0.01)
    )(ffn_output)
    x = keras.layers.Add()([x, ffn_output])
    x = keras.layers.LayerNormalization()(x)

    # Pooling layer
    x = keras.layers.GlobalAveragePooling1D()(x)

    # Dropout layer
    x = keras.layers.Dropout(0.1)(x)

    # Output layers
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(
        dense_dim, activation="relu", kernel_regularizer=keras.regularizers.l2(0.01)
    )(x)
    outputs = keras.layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

In [ ]:
def train_transformer_model(
    X_train: pd.DataFrame,
    Y_train: pd.Series,
    X_val: pd.DataFrame,
    Y_val: pd.Series,
    epochs: int = 10,
    batch_size: int = 64,
    verbose: int = 1,
):
    """
    Train a transformer model for readmission prediction.

    Args:
        X_train: Training features
        Y_train: Training target
        X_val: Validation features
        Y_val: Validation target
        epochs: Number of training epochs
        batch_size: Batch size for training
        verbose: Verbosity level (0, 1, or 2)

    Returns:
        Tuple of (trained model, training history)
    """

    early_stopping_callback = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        min_delta=0.001,
        patience=5,
        verbose=1,
        mode="min",
        restore_best_weights=True,
    )

    input_shape = X_train.shape[1]
    num_classes = len(np.unique(Y_train))

    model = build_transformer_model(input_shape, num_classes)

    # Compile model
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    # Fit model
    history = model.fit(
        X_train,
        Y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_val, Y_val),
        callbacks=[early_stopping_callback],
        verbose=verbose,
    )

    return model, history

In [ ]:
def plot_training_history(history, title_prefix="Model"):
    """
    Plot training and validation accuracy and loss over epochs.

    Args:
        history: Keras History object from model training
        title_prefix: Prefix for plot titles
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Plot accuracy
    ax1.plot(history.history["accuracy"], label="Train Accuracy")
    ax1.plot(history.history["val_accuracy"], label="Val Accuracy")
    ax1.set_title(f"{title_prefix} Accuracy")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy")
    ax1.legend()
    ax1.grid(alpha=0.3)

    # Plot loss
    ax2.plot(history.history["loss"], label="Train Loss")
    ax2.plot(history.history["val_loss"], label="Val Loss")
    ax2.set_title(f"{title_prefix} Loss")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Train the transformer model on Rahil's data splits
model_rahil, history_rahil = train_transformer_model(
    X_train=df_train[df_train.columns.difference(["readmitted"])],
    Y_train=df_train["readmitted"],
    X_val=df_val[df_val.columns.difference(["readmitted"])],
    Y_val=df_val["readmitted"],
    epochs=10,
    batch_size=64,
    verbose=1,
)

### 3.1 Train on Rahil's Data Splits

In [ ]:
# Plot training history for Rahil's data
plot_training_history(history_rahil)

In [ ]:
# Evaluate on Rahil's test set
test_loss_rahil, test_accuracy_rahil = model_rahil.evaluate(
    df_test[df_test.columns.difference(["readmitted"])],
    df_test["readmitted"],
    verbose=1,
)
print(
    f"\nRahil's Data - Test Loss: {test_loss_rahil:.4f}, Test Accuracy: {test_accuracy_rahil:.4f}"
)

### 3.2 Train on Colin's Dev Data Splits

In [ ]:
# Train the transformer model on Colin's dev data splits
model_colin_dev, history_colin_dev = train_transformer_model(
    X_train=df_train_dev,
    Y_train=df_y_train_dev.values.ravel(),
    X_val=df_val_dev,
    Y_val=df_y_val_dev.values.ravel(),
    epochs=10,
    batch_size=64,
    verbose=1,
)

In [ ]:
# Plot training history for Colin's dev data
plot_training_history(history_colin_dev)

In [ ]:
# Evaluate on Colin's dev test set
test_loss_colin_dev, test_accuracy_colin_dev = model_colin_dev.evaluate(
    df_test_dev, df_y_test_dev.values.ravel(), verbose=1
)
print(
    f"\nColin's Dev Data - Test Loss: {test_loss_colin_dev:.4f}, Test Accuracy: {test_accuracy_colin_dev:.4f}"
)

### 3.3 Compare Results: Rahil's vs Colin's Data Processing

In [ ]:
# Calculate metrics for both models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# Predictions for Rahil's data
Y_pred_rahil_proba = model_rahil.predict(
    df_test[df_test.columns.difference(["readmitted"])]
)
Y_pred_rahil = (Y_pred_rahil_proba > 0.5).astype(int).ravel()
Y_true_rahil = df_test["readmitted"].values

# Predictions for Colin's dev data
Y_pred_colin_dev_proba = model_colin_dev.predict(df_test_dev)
Y_pred_colin_dev = (Y_pred_colin_dev_proba > 0.5).astype(int).ravel()
Y_true_colin_dev = df_y_test_dev.values.ravel()

# Calculate metrics
metrics_comparison = pd.DataFrame(
    {
        "Data Source": ["Rahil's Processing", "Colin's Dev Processing"],
        "Test Accuracy": [
            accuracy_score(Y_true_rahil, Y_pred_rahil),
            accuracy_score(Y_true_colin_dev, Y_pred_colin_dev),
        ],
        "Precision": [
            precision_score(Y_true_rahil, Y_pred_rahil),
            precision_score(Y_true_colin_dev, Y_pred_colin_dev),
        ],
        "Recall": [
            recall_score(Y_true_rahil, Y_pred_rahil),
            recall_score(Y_true_colin_dev, Y_pred_colin_dev),
        ],
        "F1 Score": [
            f1_score(Y_true_rahil, Y_pred_rahil),
            f1_score(Y_true_colin_dev, Y_pred_colin_dev),
        ],
        "Test Loss": [test_loss_rahil, test_loss_colin_dev],
    }
)

print("\n" + "=" * 70)
print("TRANSFORMER MODEL COMPARISON: Rahil's vs Colin's Data Processing")
print("=" * 70)
print(metrics_comparison.to_string(index=False))
print("=" * 70)

In [ ]:
# Visualize comparison with bar charts
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = ["Test Accuracy", "Precision", "Recall", "F1 Score"]
colors = ["#1f77b4", "#ff7f0e"]

for idx, metric in enumerate(metrics_to_plot):
    row = idx // 2
    col = idx % 2
    ax = axes[row, col]

    values = metrics_comparison[metric].values
    bars = ax.bar(metrics_comparison["Data Source"], values, color=colors)

    ax.set_title(f"{metric} Comparison", fontsize=12, fontweight="bold")
    ax.set_ylabel(metric, fontsize=10)
    ax.set_ylim([0, 1])
    ax.grid(axis="y", alpha=0.3)

    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            height,
            f"{height:.4f}",
            ha="center",
            va="bottom",
            fontsize=10,
        )

plt.tight_layout()
plt.suptitle(
    "Transformer Model Performance: Data Processing Comparison",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.show()

In [ ]:
# Compare confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix for Rahil's data
cm_rahil = confusion_matrix(Y_true_rahil, Y_pred_rahil)
im1 = axes[0].imshow(cm_rahil, cmap="Blues", interpolation="nearest")
axes[0].set_title("Confusion Matrix - Rahil's Data", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["No Readmit", "Readmit"])
axes[0].set_yticklabels(["No Readmit", "Readmit"])
for i in range(2):
    for j in range(2):
        axes[0].text(
            j,
            i,
            str(cm_rahil[i, j]),
            ha="center",
            va="center",
            color="white" if cm_rahil[i, j] > cm_rahil.max() / 2 else "black",
            fontsize=14,
            fontweight="bold",
        )
plt.colorbar(im1, ax=axes[0])

# Confusion matrix for Colin's dev data
cm_colin_dev = confusion_matrix(Y_true_colin_dev, Y_pred_colin_dev)
im2 = axes[1].imshow(cm_colin_dev, cmap="Oranges", interpolation="nearest")
axes[1].set_title("Confusion Matrix - Colin's Dev Data", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
axes[1].set_xticks([0, 1])
axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(["No Readmit", "Readmit"])
axes[1].set_yticklabels(["No Readmit", "Readmit"])
for i in range(2):
    for j in range(2):
        axes[1].text(
            j,
            i,
            str(cm_colin_dev[i, j]),
            ha="center",
            va="center",
            color="white" if cm_colin_dev[i, j] > cm_colin_dev.max() / 2 else "black",
            fontsize=14,
            fontweight="bold",
        )
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

### 3.4 Class Imbalance Analysis

**Recommendations to Address Class Imbalance:**

1. **Use Class Weights**: Add `class_weight` parameter to model compilation to penalize misclassifications of minority class more heavily
2. **Adjust Decision Threshold**: Instead of 0.5, use a lower threshold (e.g., 0.3) to classify more samples as positive
3. **Oversampling/Undersampling**: Use SMOTE or other techniques to balance the training data
4. **Use Different Loss Function**: Try focal loss which focuses more on hard-to-classify examples
5. **Optimize for F1 Score**: Focus on maximizing F1 or recall rather than just accuracy

## 4. Notebook Exports

#### 4.1 Export models

In [ ]:
# Export the transformer model (trained on Rahil's data) to results/
model_rahil.save("../results/transformer.h5")
print("Model saved to ../results/transformer.h5")

#### 4.2 Export visualizations

In [ ]:
# Create visualizations only for combined cleaned data (which has all features)
kde_plots_combined = create_kde_plots(df_combined_clean, continuous_features)
box_plots_combined = create_box_plots(df_combined_clean, continuous_features)
violin_combined = create_violin_plots(df_combined_clean, continuous_features)

# Save visualizations to results directory
kde_plots_combined.save("../results/kde_plots_combined.html")
box_plots_combined.save("../results/box_plots_combined.html")
violin_combined.save("../results/violin_plots_combined.html")
print("Visualizations saved successfully!")